# LiDAR preprocessing

Converts every raw LiDAR point cloud in `base_station/_out_lidar/` into a 256 x 256 bird's-eye-view histogram saved in `base_station/preprocessed_out_lidar/`. Run this notebook from the repository root.

## Helper functions

In [ ]:
def rotate_points_around_y(points, theta_deg):
    """
    points: NumPy array of shape (N, 3) or (N, 4)
            [x, y, z] (or [x, y, z, intensity])
    theta_deg: Angle to rotate around the y-axis (in degrees, positive value means counterclockwise)

    return: Rotated points (N, 3) or (N, 4)
    """
    # 1) Convert degrees to radians
    theta = np.radians(theta_deg)

    # 2) Rotation matrix around the y-axis (3x3)
    #    R_y(theta) = [[ cosθ,  0, sinθ],
    #                   [ 0,     1,   0 ],
    #                   [-sinθ,  0, cosθ]]
    rot_mat_y = np.array([
        [ np.cos(theta),  0, np.sin(theta)],
        [            0,   1,            0 ],
        [-np.sin(theta),  0, np.cos(theta)]
    ])

    # 3) Separate processing depending on whether points are (N, 3) or (N, 4)
    if points.shape[1] == 3:
        # Rotate (x, y, z) part only
        # (N, 3) x (3, 3)^T -> (N, 3)
        rotated_xyz = points @ rot_mat_y.T
        return rotated_xyz
    
    elif points.shape[1] == 4:
        # First 3 are (x, y, z), the rest are intensity, etc.
        xyz = points[:, :3]
        other = points[:, 3:]  # e.g., intensity or other values
        
        rotated_xyz = xyz @ rot_mat_y.T
        # Concatenate the rotated part with the rest
        rotated_points = np.hstack((rotated_xyz, other))
        return rotated_points
    
    else:
        raise ValueError("points shape must be (N, 3) or (N, 4).")

def filter_points_by_height(points):
    """
    points: Point cloud of shape (N, 3) or (N, 4)
    z_min, z_max: Height range to keep
    return: Points within the range z_min <= z <= z_max
    """
    # Assume points[:, 2] are the z values of each point
    # print(f"minimum z value: {min(points[:,2])}")
    # print(f"minimum z value: {max(points[:,2])}")
    mask = (points[:, 2] >= min(points[:, 2]) + 0.0) & (points[:, 2] <= max(points[:, 2]) - 50)
    return points[mask]

def lidar_to_histogram_features(lidar, _kind):
    """
    Convert LiDAR point cloud into 2-bin histogram over 256x256 grid
    LiDAR data seems to be x position inversed 
    """
    # Rotate the point cloud about the y axis (40 deg for the 2-lane base station setup)
    trans_lidar = rotate_points_around_y(lidar, 40)
    trans_lidar = filter_points_by_height(trans_lidar)
    def splat_points(point_cloud):
        # 256 x 256 grid
        hist_max_per_pixel = 10
        xbins = np.linspace(0, 150, 257)
        ybins = np.linspace(-150, 150, 257)
        
        hist = np.histogramdd(point_cloud[...,:2], bins=(xbins, ybins))[0]
        hist[hist > hist_max_per_pixel] = hist_max_per_pixel
        overhead_splat = hist / hist_max_per_pixel
        return overhead_splat
    lidar_feature = splat_points(trans_lidar)
    lidar_feature = lidar_feature[np.newaxis, :, :]
    return lidar_feature

## Convert all LiDAR frames

In [ ]:
from pathlib import Path
import numpy as np
import open3d as o3d

# 1) Define your paths as Path objects (run this notebook from the repository root)
root        = Path("base_station").resolve()
in_base_lid = root / "_out_lidar"
out_base    = root / "preprocessed_out_lidar"
out_base.mkdir(parents=True, exist_ok=True)

# 2) Collect all raw LiDAR frames, e.g. …/base_station/_out_lidar/episode_1/001757.ply
ply_paths = sorted(in_base_lid.glob("*/*.ply"))

for ply_path in ply_paths:
    # 3) Compute the sub-folder under _out_lidar (e.g. "episode_1")
    rel = ply_path.parent.relative_to(in_base_lid)

    # 4) Make the matching folder under preprocessed_out_lidar
    target_dir = out_base / rel
    target_dir.mkdir(parents=True, exist_ok=True)

    # 5) Read, convert, and save into that folder
    pts  = np.asarray(o3d.io.read_point_cloud(ply_path).points, dtype=np.float32)
    hist = lidar_to_histogram_features(pts, root.name)      # (1,256,256)
    out_file = target_dir / f"{ply_path.stem}.npy"
    np.save(out_file, hist.astype(np.float16))

print(f"saved {len(ply_paths)} LiDAR maps → {out_base}")

## Optional: PyTorch `Dataset` for the generated data

Example loader for `dataset_index_local.csv` (created by `make_data_csv.ipynb`). Not needed for dataset generation; requires `torch`, `torchvision`, `pillow` and `scipy`.

In [ ]:
from PIL import Image
import pandas as pd
import numpy as np
import torch 
from torch.utils.data import Dataset
import open3d as o3d
from scipy.io import loadmat
import math
import copy
from torchvision import transforms
import time

class MultimodalDataLoader(Dataset):
    def __init__(self, root, root_csv, test=False):
        self.dataframe = pd.read_csv(root + root_csv)
        self.root = root
        # self.seq_len = 5
        self.test = test
        self.gps_num = 26 #maximum number of observed vehicles
        # quick sanity check: every RSS file has same length
        first_len = len(pd.read_csv(self.dataframe['RSS'][0][0:800]))
        assert all(len(pd.read_csv(p)) == first_len for p in self.dataframe['RSS'])
               #print("Inconsistent RSS lengths – clean step incomplete?")
        self.img_tf = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),                         # → float32 , 0‑1
            transforms.Normalize(mean=[0.485,0.456,0.406], # ImageNet stats
                        std=[0.229,0.224,0.225]),
        ])
    def __len__(self):
        """Returns the length of the dataset."""
        return self.dataframe.shape[0]

    def __getitem__(self, index):
        """Returns the item at index idx."""
        data = dict()
        # data['rgbs'] = []       # [seq_len, batch_size, channel, 256, 256] 
        # data['lidars'] = []     # [seq_len, batch_size, channel, 256, 256] 
        # data['radars'] = []     # [seq_len, batch_size, points, attribution]
        # data['gpss'] = []       # [seq_len, batch_size, vehicle_num, pos] 
        # data['scenario'] = []
        # data['rss'] = []
        # data['hasLoS'] = []

        # instanceidx = ['1', '2', '3', '4', '5'] # 5 time instances
        # self.seq_len = len(instanceidx)

        data['scenario'] = self.dataframe['Scene'][index]

        t0 = time.time()

        gpsdataframe = pd.read_csv(self.dataframe['GPS'][index])
        bspos = [0, 0, 0]
        rotate = 0
        if data['scenario'] == 'base_station':
            bspos = [26.252628, 86.328842, 5] #This is MATLAB coordinate
            # rotate = 3 * math.pi / 2
        matrix = gpsdataframe.to_numpy()
        vehicles = []
        for mat in matrix:
            tmpPt = (mat[3:6] - bspos).tolist() #mat[2:5] are the x, y, z coordinates in csv files 
            # tmpPt[0], tmpPt[1] = rotate_point(tmpPt[0], tmpPt[1], rotate) #But, why do we need rotate?
            vehicles.append(tmpPt)
        for _ in range(self.gps_num - len(matrix)): #zero padding to another vehicles 
            vehicles.append([0, 0, 0])
        # data['gpss'].append(torch.from_numpy(np.array(vehicles))) #
        data['gpss'] = torch.from_numpy(np.array(vehicles, dtype=np.float32))
        t1 = time.time()

        # RGB data
        rgb = Image.open(self.dataframe['RGB'][index]).convert('RGB')
        rgb = self.img_tf(rgb)        
        data['rgbs'] = rgb                          # [3,256,256]
        # imgs = np.array(Image.open(self.dataframe['RGB'][index]).resize((256,256)).convert("RGB")
        # data['rgbs'].append(torch.from_numpy(np.transpose(imgs, (2, 0, 1))))
        # data['rgbs'] = torch.from_numpy(np.transpose(imgs, (2, 0, 1)))
        t2 = time.time()
        # Radar data
        rds = np.load(self.dataframe['Radar'][index])
        # data['radars'].append(radar_features(rds))
        radar_np = radar_features(rds)
        radar_tensor = torch.from_numpy(radar_np).float()
        data['radars'] = radar_tensor
        t3 = time.time()
        # LiDAR data
        PT_transformed = np.load(self.dataframe['LiDAR'][index]).astype(np.float32)  # preprocessed (1,256,256)
        # data['lidars'].append(PT)
        PT_transformed = torch.from_numpy(PT_transformed).float()
        data['lidars'] = PT_transformed
        t4 = time.time()

        rssdf = pd.read_csv(self.dataframe['RSS'][index])
        
        # RSS value (target)       (N,)   float32
        data['rss']     = torch.as_tensor(
            rssdf['RSS_dBm'].values,
            dtype=torch.float32)
        
        data['rss_los']     = torch.as_tensor(
            rssdf['RSS_LoS_dBm'].values,
            dtype=torch.float32)
        
        # LoS indicator            (N,)   bool → float32
        data['hasLoS']  = torch.as_tensor(
            rssdf['hasLoS'].values.astype(np.float32))
        t5 = time.time()


        if index == 1:  # print once
            print(f"gps  : {t1-t0:.3f}s")
            print(f"rgb  : {t2-t1:.3f}s")
            print(f"radar: {t3-t2:.3f}s")
            print(f"lidar: {t4-t3:.3f}s")
            print(f"RSS: {t5-t4:.3f}s")

        return data
      
def rotate_points_around_y(points, theta_deg):
    """
    points: NumPy array of shape (N, 3) or (N, 4)
            [x, y, z] (or [x, y, z, intensity])
    theta_deg: Angle to rotate around the y-axis (in degrees, positive value means counterclockwise)

    return: Rotated points (N, 3) or (N, 4)
    """
    # 1) Convert degrees to radians
    theta = np.radians(theta_deg)

    # 2) Rotation matrix around the y-axis (3x3)
    #    R_y(theta) = [[ cosθ,  0, sinθ],
    #                   [ 0,     1,   0 ],
    #                   [-sinθ,  0, cosθ]]
    rot_mat_y = np.array([
        [ np.cos(theta),  0, np.sin(theta)],
        [            0,   1,            0 ],
        [-np.sin(theta),  0, np.cos(theta)]
    ])

    # 3) Separate processing depending on whether points are (N, 3) or (N, 4)
    if points.shape[1] == 3:
        # Rotate (x, y, z) part only
        # (N, 3) x (3, 3)^T -> (N, 3)
        rotated_xyz = points @ rot_mat_y.T
        return rotated_xyz
    
    elif points.shape[1] == 4:
        # First 3 are (x, y, z), the rest are intensity, etc.
        xyz = points[:, :3]
        other = points[:, 3:]  # e.g., intensity or other values
        
        rotated_xyz = xyz @ rot_mat_y.T
        # Concatenate the rotated part with the rest
        rotated_points = np.hstack((rotated_xyz, other))
        return rotated_points
    
    else:
        raise ValueError("points shape must be (N, 3) or (N, 4).")

def filter_points_by_height(points):
    """
    points: Point cloud of shape (N, 3) or (N, 4)
    z_min, z_max: Height range to keep
    return: Points within the range z_min <= z <= z_max
    """
    # Assume points[:, 2] are the z values of each point
    # print(f"minimum z value: {min(points[:,2])}")
    # print(f"minimum z value: {max(points[:,2])}")
    mask = (points[:, 2] >= min(points[:, 2]) + 0.0) & (points[:, 2] <= max(points[:, 2]) - 50)
    return points[mask]

def lidar_to_histogram_features(lidar, _kind):
    """
    Convert LiDAR point cloud into 2-bin histogram over 256x256 grid
    LiDAR data seems to be x position inversed 
    """
    # Rotate the point cloud about the y axis (40 deg for the 2-lane base station setup)
    trans_lidar = rotate_points_around_y(lidar, 40)
    trans_lidar = filter_points_by_height(trans_lidar)
    def splat_points(point_cloud):
        # 256 x 256 grid
        hist_max_per_pixel = 10
        xbins = np.linspace(0, 150, 257)
        ybins = np.linspace(-150, 150, 257)
        
        hist = np.histogramdd(point_cloud[...,:2], bins=(xbins, ybins))[0]
        hist[hist > hist_max_per_pixel] = hist_max_per_pixel
        overhead_splat = hist / hist_max_per_pixel
        return overhead_splat
    lidar_feature = splat_points(trans_lidar)
    lidar_feature = lidar_feature[np.newaxis, :, :]
    return lidar_feature

def radar_features(radar):
    fixed_points = 300
    velocity, azimuth, altitude, depth = radar[:, 0], radar[:, 1], radar[:, 2], radar[:, 3]

    # Sort by altitude
    sorted_indices = np.argsort(altitude)  # Sorted indices by ascending altitude
    if len(radar) > fixed_points:
        sorted_indices = sorted_indices[:fixed_points]  # Select top 300
    else:
        if len(radar) < fixed_points:
            # Pad with duplicates if not enough points
            sorted_indices = np.pad(sorted_indices, (0, fixed_points - len(radar)), mode='wrap')

    # Select sorted data
    velocity = velocity[sorted_indices]
    azimuth = azimuth[sorted_indices]
    altitude = altitude[sorted_indices]
    depth = depth[sorted_indices]
    return np.array([velocity, azimuth, altitude, depth]).T

def rotate_point(x, y, theta):
    """
    Rotate point (x, y) around the origin (0,0) by angle theta (in radians).
    """
    # Apply rotation transformation
    # x' = x*cos(theta) - y*sin(theta)
    # y' = x*sin(theta) + y*cos(theta)
    x_rotated = x * math.cos(theta) - y * math.sin(theta)
    y_rotated = x * math.sin(theta) + y * math.cos(theta)
    return x_rotated, y_rotated